In [8]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s4e11')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/playground-series-s4e11


In [9]:
import os

print(os.listdir(path))   # xem có file gì

['sample_submission.csv', 'train.csv', 'test.csv']


In [10]:
import pandas as pd


# đọc từng csv và in thông tin
for file in os.listdir(path):
    if file.endswith(".csv"):
        file_path = os.path.join(path, file)

        print(f"\nFILE: {file}")

        df = pd.read_csv(file_path)

        print("\nShape:")
        print(df.shape)

        print("\nColumns:")
        print(df.columns.tolist())

        print("\nDtypes:")
        print(df.dtypes)

        print("\nMissing values:")
        print(df.isnull().sum())

        print("\nStatistics:")
        print(df.describe(include="all"))

        print("\nFirst 5 rows:")
        print(df.head())

        print("\n" + "=" * 50)


FILE: sample_submission.csv

Shape:
(93800, 2)

Columns:
['id', 'Depression']

Dtypes:
id            int64
Depression    int64
dtype: object

Missing values:
id            0
Depression    0
dtype: int64

Statistics:
                  id  Depression
count   93800.000000     93800.0
mean   187599.500000         0.0
std     27077.871962         0.0
min    140700.000000         0.0
25%    164149.750000         0.0
50%    187599.500000         0.0
75%    211049.250000         0.0
max    234499.000000         0.0

First 5 rows:
       id  Depression
0  140700           0
1  140701           0
2  140702           0
3  140703           0
4  140704           0


FILE: train.csv

Shape:
(140700, 20)

Columns:
['id', 'Name', 'Gender', 'Age', 'City', 'Working Professional or Student', 'Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Finan

In [11]:
train_df = pd.read_csv(f"{path}/train.csv")
test_df = pd.read_csv(f"{path}/test.csv")

In [12]:
df = train_df.copy()
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    print(f"\n===== {col} =====")
    print(df[col].unique())


===== Name =====
['Aaradhya' 'Vivan' 'Yuvraj' 'Rhea' 'Vani' 'Ritvik' 'Rajveer' 'Aishwarya'
 'Simran' 'Utkarsh' 'Aahana' 'Tejas' 'Aadhya' 'Kiran' 'Aditi' 'Suhani'
 'Jiya' 'Bhavesh' 'Armaan' 'Ishaani' 'Prachi' 'Pratyush' 'Abhinav'
 'Siddhesh' 'Aditya' 'Aarav' 'Asha' 'Kashish' 'Prisha' 'Chhavi' 'Tanmay'
 'Vihaan' 'Shiv' 'Anvi' 'Darsh' 'Samar' 'Raunak' 'Mahi' 'Shaurya' 'Vidya'
 'Jai' 'Ayush' 'Ansh' 'Anand' 'Yashvi' 'Shrey' 'Ritika' 'Mihir' 'Isha'
 'Arjun' 'Rohan' 'Pratham' 'Nirvaan' 'Ishaan' 'Aarya' 'Riya' 'Aariv'
 'Raghavendra' 'Mahika' 'Abhishek' 'Harshil' 'Janvi' 'Kartikeya' 'Shivam'
 'Advait' 'Reyansh' 'Saanvi' 'Ivaan' 'Pallavi' 'Sneha' 'Ayaan' 'Aakash'
 'Raghav' 'Satyam' 'Aarush' 'Vibha' 'Rupal' 'Sanya' 'Mira' 'Rashi' 'Shlok'
 'Harsha' 'Divya' 'Pranav' 'Hrithik' 'Tushar' 'Garima' 'Zoya' 'Kian'
 'Navya' 'Lakshay' 'Kriti' 'Palak' 'Aryan' 'Parth' 'Ishan' 'Rupak'
 'Atharv' 'Aarti' 'Anirudh' 'Kabir' 'Sanjeev' 'Sanket' 'Tara' 'Gagan'
 'Anjali' 'Gaurav' 'Vikram' 'Yogesh' 'Ila' 'Rishi' 'Ayan

In [13]:
# ===== ORDINAL ENCODING =====
sleep_map = {
    "Less than 5 hours": 0,
    "5-6 hours": 1,
    "6-7 hours": 2,
    "7-8 hours": 3,
    "8-9 hours": 4,
    "More than 8 hours": 5,
}

diet_map = {
    "Unhealthy": 0,
    "Moderate": 1,
    "Healthy": 2,
    "More Healthy": 3,
}

binary_map = {
    "No": 0,
    "Yes": 1,
}
def preprocessing_df(_df):

    
    _df["Sleep Duration"] = _df["Sleep Duration"].map(sleep_map)
    
    _df["Dietary Habits"] = _df["Dietary Habits"].map(diet_map)
    
    _df["Have you ever had suicidal thoughts ?"] = (
        _df["Have you ever had suicidal thoughts ?"]
        .map(binary_map)
    )
    
    _df["Family History of Mental Illness"] = (
        _df["Family History of Mental Illness"]
        .map(binary_map)
    )
    
    # ===== ONE HOT ENCODING =====
    
    onehot_cols = [
        "Gender",
        "Working Professional or Student",
    ]
    
    _df = pd.get_dummies(
        _df,
        columns=onehot_cols,
        drop_first=True,
    )
    
    # ===== FREQUENCY ENCODING =====
    # tránh explode feature
    
    freq_cols = [
        "City",
        "Profession",
        "Degree",
    ]
    
    _df = _df.drop(columns=["Name", "id"])
    
    for col in freq_cols:
    
        freq = _df[col].value_counts()
    
        rare = freq[freq < 10].index
        _df[col] = _df[col].fillna("Other")
        _df.loc[_df[col].isin(rare), col] = "Other"
    
    for col in freq_cols:
        freq = _df[col].value_counts()
    
        _df[col] = _df[col].map(freq)
    return _df
df = preprocessing_df(df)

In [14]:
import numpy as np
df["Sleep Duration"] = (
    df["Sleep Duration"]
    .where(df["Sleep Duration"].isin(sleep_map), np.nan)
    .fillna(df["Sleep Duration"].mode()[0])
    .map(sleep_map)
)

df["Dietary Habits"] = (
    df["Dietary Habits"]
    .where(df["Dietary Habits"].isin(diet_map), np.nan)
    .fillna(df["Dietary Habits"].mode()[0])
    .map(diet_map)
)

df["Financial Stress"] = (
    df["Financial Stress"]
    .fillna(df["Financial Stress"].median())
)

In [15]:
print("\nMissing values:")
print(df.isnull().sum())



Missing values:
Age                                                          0
City                                                         0
Profession                                                   0
Academic Pressure                                       112803
Work Pressure                                            27918
CGPA                                                    112802
Study Satisfaction                                      112803
Job Satisfaction                                         27910
Sleep Duration                                          140700
Dietary Habits                                          140700
Degree                                                       0
Have you ever had suicidal thoughts ?                        0
Work/Study Hours                                             0
Financial Stress                                             0
Family History of Mental Illness                             0
Depression                            

In [16]:
df.columns

Index(['Age', 'City', 'Profession', 'Academic Pressure', 'Work Pressure',
       'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration',
       'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?',
       'Work/Study Hours', 'Financial Stress',
       'Family History of Mental Illness', 'Depression', 'Gender_Male',
       'Working Professional or Student_Working Professional'],
      dtype='object')

In [17]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ===== split =====

X = df.drop(columns=["Depression"])
y = df["Depression"]

neg = (y == 0).sum()
pos = (y == 1).sum()

scale = neg / pos


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# ===== model =====

model = XGBClassifier(
    n_estimators=3000,
    max_depth=6,
    learning_rate=0.03,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=3,
    gamma=0.1,

    reg_alpha=0.1,
    reg_lambda=1.0,

    scale_pos_weight=scale,

    objective="binary:logistic",
    eval_metric="logloss",

    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
)

# ===== train =====

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=100,
)



[0]	validation_0-logloss:0.67219
[100]	validation_0-logloss:0.23034
[200]	validation_0-logloss:0.21242
[300]	validation_0-logloss:0.20871
[400]	validation_0-logloss:0.20712
[500]	validation_0-logloss:0.20577
[600]	validation_0-logloss:0.20455
[700]	validation_0-logloss:0.20365
[800]	validation_0-logloss:0.20289
[900]	validation_0-logloss:0.20225
[1000]	validation_0-logloss:0.20180
[1100]	validation_0-logloss:0.20126
[1200]	validation_0-logloss:0.20086
[1300]	validation_0-logloss:0.20020
[1400]	validation_0-logloss:0.19988
[1500]	validation_0-logloss:0.19966
[1600]	validation_0-logloss:0.19924
[1700]	validation_0-logloss:0.19895
[1800]	validation_0-logloss:0.19865
[1900]	validation_0-logloss:0.19837
[2000]	validation_0-logloss:0.19822
[2100]	validation_0-logloss:0.19806
[2200]	validation_0-logloss:0.19790
[2300]	validation_0-logloss:0.19787
[2400]	validation_0-logloss:0.19774
[2500]	validation_0-logloss:0.19761
[2600]	validation_0-logloss:0.19767
[2700]	validation_0-logloss:0.19766
[280

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=3000, n_jobs=-1,
              num_parallel_tree=None, ...)

In [18]:
# ===== predict =====

pred = model.predict(X_valid)

# ===== metrics =====

acc = accuracy_score(y_valid, pred)
f1 = f1_score(y_valid, pred)

print("Accuracy:", acc)
print("F1:", f1)
print(classification_report(y_valid, pred))

Accuracy: 0.9216062544420753
F1: 0.8046404534183492
              precision    recall  f1-score   support

           0       0.97      0.93      0.95     23027
           1       0.74      0.89      0.80      5113

    accuracy                           0.92     28140
   macro avg       0.85      0.91      0.88     28140
weighted avg       0.93      0.92      0.92     28140



/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [05:13:42] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [24]:

def preprocess(df):

    df = df.copy()
    drop_cols = ["Name", "id"]

    existing_drop_cols = [
        col for col in drop_cols
        if col in df.columns
    ]

    df = df.drop(columns=existing_drop_cols)


    freq_cols = [
        "City",
        "Profession",
        "Degree",
    ]

    for col in freq_cols:

        freq = df[col].value_counts()

        rare = freq[freq < 10].index

        df[col] = df[col].fillna("Other")

        df.loc[
            df[col].isin(rare),
            col
        ] = "Other"


    df["Sleep Duration"] = (
        df["Sleep Duration"]
        .where(
            df["Sleep Duration"].isin(sleep_map),
            np.nan
        )
        .fillna(df["Sleep Duration"].mode()[0])
        .map(sleep_map)
    )


    df["Dietary Habits"] = (
        df["Dietary Habits"]
        .where(
            df["Dietary Habits"].isin(diet_map),
            np.nan
        )
        .fillna(df["Dietary Habits"].mode()[0])
        .map(diet_map)
    )

    df["Financial Stress"] = (
        df["Financial Stress"]
        .fillna(df["Financial Stress"].median())
    )

    return df

cat_df = train_df.copy()

cat_df = preprocess(cat_df)

In [25]:
from catboost import CatBoostClassifier
X = cat_df.drop(columns=["Depression"])
y = cat_df["Depression"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

neg = (y == 0).sum()
pos = (y == 1).sum()

weight = neg / pos
# ===== categorical columns =====
# CatBoost xử lý categorical native

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

# index của categorical columns
cat_features = [
    X_train.columns.get_loc(col)
    for col in cat_cols
]

# ===== model =====

model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,

    loss_function="Logloss",
    eval_metric="F1",
    l2_leaf_reg=3,
    # class_weights=[1, 4.5],
    random_seed=42,

    task_type="GPU",   # bỏ dòng này nếu train CPU

    verbose=100,
)

# ===== train =====

model.fit(
    X_train,
    y_train,

    cat_features=cat_features,

    eval_set=(X_valid, y_valid),

    use_best_model=True,
)



0:	learn: 0.7811177	test: 0.7797066	best: 0.7797066 (0)	total: 29.5ms	remaining: 59s
100:	learn: 0.8220598	test: 0.8195866	best: 0.8199518 (97)	total: 2.04s	remaining: 38.4s
200:	learn: 0.8299092	test: 0.8262610	best: 0.8266613 (197)	total: 4.05s	remaining: 36.2s
300:	learn: 0.8336198	test: 0.8270135	best: 0.8272618 (213)	total: 6.05s	remaining: 34.2s
400:	learn: 0.8358492	test: 0.8283172	best: 0.8283172 (400)	total: 8.11s	remaining: 32.3s
500:	learn: 0.8382807	test: 0.8282889	best: 0.8291854 (439)	total: 10.1s	remaining: 30.3s
600:	learn: 0.8393506	test: 0.8285087	best: 0.8291854 (439)	total: 12.1s	remaining: 28.3s
700:	learn: 0.8412192	test: 0.8284260	best: 0.8291854 (439)	total: 14.1s	remaining: 26.1s
800:	learn: 0.8429781	test: 0.8287911	best: 0.8291854 (439)	total: 16s	remaining: 23.9s
900:	learn: 0.8448894	test: 0.8288935	best: 0.8291854 (439)	total: 17.9s	remaining: 21.8s
1000:	learn: 0.8458341	test: 0.8293607	best: 0.8293607 (999)	total: 19.8s	remaining: 19.7s
1100:	learn: 0.84

CatBoostClassifier(depth=6, eval_metric='F1', iterations=2000, l2_leaf_reg=3, learning_rate=0.03, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [26]:
# ===== predict =====

prob = model.predict(X_valid)

# ===== report =====

print(classification_report(y_valid, pred))

              precision    recall  f1-score   support

           0       0.97      0.93      0.95     23027
           1       0.74      0.89      0.80      5113

    accuracy                           0.92     28140
   macro avg       0.85      0.91      0.88     28140
weighted avg       0.93      0.92      0.92     28140



Train with full ds

In [27]:
X_full = cat_df.drop(columns=["Depression"])
y_full = cat_df["Depression"]

model.fit(
    X_full,
    y_full,
    cat_features=cat_features,
)

0:	learn: 0.7808355	total: 28.6ms	remaining: 57.1s
100:	learn: 0.8214694	total: 1.95s	remaining: 36.7s
200:	learn: 0.8291767	total: 3.83s	remaining: 34.3s
300:	learn: 0.8317207	total: 5.72s	remaining: 32.3s
400:	learn: 0.8339779	total: 7.64s	remaining: 30.5s
500:	learn: 0.8360744	total: 9.6s	remaining: 28.7s
600:	learn: 0.8373836	total: 11.5s	remaining: 26.7s
700:	learn: 0.8386094	total: 13.3s	remaining: 24.7s
800:	learn: 0.8402232	total: 15.2s	remaining: 22.8s
900:	learn: 0.8416584	total: 17.1s	remaining: 20.9s
1000:	learn: 0.8426612	total: 19.1s	remaining: 19s
1100:	learn: 0.8441983	total: 20.9s	remaining: 17.1s
1200:	learn: 0.8452405	total: 22.8s	remaining: 15.2s
1300:	learn: 0.8464803	total: 24.7s	remaining: 13.2s
1400:	learn: 0.8474921	total: 26.5s	remaining: 11.3s
1500:	learn: 0.8490732	total: 28.5s	remaining: 9.47s
1600:	learn: 0.8497032	total: 30.4s	remaining: 7.58s
1700:	learn: 0.8508417	total: 32.3s	remaining: 5.67s
1800:	learn: 0.8519072	total: 34.2s	remaining: 3.77s
1900:	l

CatBoostClassifier(depth=6, eval_metric='F1', iterations=2000, l2_leaf_reg=3, learning_rate=0.03, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [32]:
test_df = pd.read_csv(f"{path}/test.csv")

test_ids = test_df["id"]
test_df = preprocess(test_df)
print(len(test_df))
test_pred = model.predict(test_df)
submission = pd.DataFrame({
    "id": test_ids,
    "Depression": test_pred.astype(int),
})

submission.to_csv("submission.csv", index=False)

93800


In [34]:
!kaggle competitions submit \
-c playground-series-s4e11 \
-f submission.csv \
-m "first submission"

100%|████████████████████████████████████████| 824k/824k [00:00<00:00, 1.14MB/s]
Successfully submitted to Exploring Mental Health Data